In [1]:
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm

def create_circular_electrode_mask(cx, cy, radius, X, Y):
    """
    Create a boolean mask for a circular electrode on a 2D grid.
    
    Parameters:
    - cx, cy: center coordinates in microns (physical coordinates)
    - radius: radius in microns
    - X, Y: 2D arrays of physical coordinates corresponding to the grid points
    
    Returns:
    - mask: boolean 2D array where True indicates points inside the circle
    """
    dist_from_center = np.sqrt((X - cx)**2 + (Y - cy)**2)
    mask = dist_from_center <= radius
    return mask

def create_rectangular_electrode_mask(cx, cy, width, height, X, Y):
    """
    Create a boolean mask for a rectangular electrode on a 2D grid.

    Parameters:
    - nx, ny: grid dimensions (not used but included for symmetry)
    - cx, cy: center of the rectangle (in microns)
    - width, height: size of the rectangle (in microns)
    - X, Y: 2D coordinate arrays (same shape as the grid)

    Returns:
    - mask: boolean 2D array where True indicates inside the rectangle
    """
    x1 = cx - width / 2
    x2 = cx + width / 2
    y1 = cy - height / 2
    y2 = cy + height / 2

    mask = (X >= x1) & (X <= x2) & (Y >= y1) & (Y <= y2)
    return mask

def create_rotated_hyperbolic_electrode_mask(x0, y0, a, b, theta_deg, X, Y, branch='positive'):
    """
    Create a rotated hyperbolic electrode mask centered at (x0, y0), rotated by theta_deg.

    Parameters:
    - a, b: hyperbola parameters
    - theta_deg: rotation angle in degrees
    - branch: 'positive' for u >= sqrt(1 + (bv/au)^2), 'negative' for opposite arm
    - X, Y: coordinate meshgrids
    """
    theta = np.radians(theta_deg)

    # Rotate coordinates around center
    x_shift = X - x0
    y_shift = Y - y0
    u = x_shift * np.cos(theta) + y_shift * np.sin(theta)
    v = -x_shift * np.sin(theta) + y_shift * np.cos(theta)

    lhs = (u**2) / a**2 - (v**2) / b**2

    if branch == 'positive':
        mask = lhs >= 1
    elif branch == 'negative':
        mask = lhs <= -1
    else:
        raise ValueError("branch must be 'positive' or 'negative'")

    return mask


def solve_potential(nx, ny, electrodes, fixed, V, max_iter=10000, tol=1e-3):
    """
    Solve Laplace equation on V with fixed points mask fixed and fixed potentials in V.
    Reapply fixed voltages every iteration to maintain boundary conditions.
    """
    for it in tqdm(range(max_iter)):
        V_old = V.copy()

        for y in range(1, ny - 1):
            for x in range(1, nx - 1):
                if not fixed[y, x]:
                    V[y, x] = 0.25 * (V[y + 1, x] + V[y - 1, x] + V[y, x + 1] + V[y, x - 1])

        # Reapply fixed voltages after update
        for electrode in electrodes:
            mask = electrode['mask']
            V[mask] = electrode['voltage']

        diff = np.max(np.abs(V - V_old))
        if diff < tol:
            print(f'Converged after {it} iterations.')
            break
    else:
        print(f'Max iterations reached, diff={diff}')

    return V

def compute_electric_field(V, dx=1.0, dy=1.0):
    Ey, Ex = np.gradient(V, dy, dx)  # Note numpy returns gradient as (dV/dy, dV/dx)
    return -Ex, -Ey

# Hyperbolic electrodes
# 300 x 300 micron grid with 2 micion dx
# 150 hyperbolic electrodes
# Get 0.9855 geometric factor (fit -140 to 140)
# 1e-6 tol

# %%
# User parameters
nx, ny = 150, 150         # grid points
dx = 4.0           # spacing in microns, user can change this

# Physical coordinate arrays centered at zero
x = (np.arange(nx) - nx//2) * dx
y = (np.arange(ny) - ny//2) * dx
X, Y = np.meshgrid(x, y)

# Electrode parameters (microns)
# radius = 100  # microns
# half_sep = 250 / np.sqrt(2)  # half the square side length in microns, so total 100 um apart

# # Define rod electrode centers relative to center (0,0). Rods in square geometry
# electrodes = [
#     {'center': (-half_sep,  half_sep), 'voltage': 1},   # top-left
#     {'center': ( half_sep,  half_sep), 'voltage': -1},     # top-right
#     {'center': (-half_sep, -half_sep), 'voltage': -1},     # bottom-left
#     {'center': ( half_sep, -half_sep), 'voltage': 1}    # bottom-right
# ]

# sep = 250  # Separation between electrods
# electrodes = [
#     {'center': (0,  sep), 'voltage': 1},   # top
#     {'center': (0,  -sep), 'voltage': 1},     # bottom
#     {'center': (-sep, 0), 'voltage': -1},     # left
#     {'center': (sep, 0), 'voltage': -1}    # right
# ]

# Define rectangular electrode centers relative to center (0,0)
# width = 60
# height = 4.6
# x_sep = 60
# y_sep = 4 + 2.3
# electrodes = [
#     {'center': (-x_sep,  y_sep), 'dims': (width, height), 'voltage': 1},   # top-left
#     {'center': ( x_sep,  y_sep), 'dims': (width, height), 'voltage': 0},     # top-right
#     {'center': (-x_sep, -y_sep), 'dims': (width, height), 'voltage': 0},     # bottom-left
#     {'center': ( x_sep, -y_sep), 'dims': (width, height), 'voltage': 1}    # bottom-right
# ]

# Define rectangular electrode centers for surface trap
# width = 40
# height = 5
# x_sep = 20
# outer_x_sep = 80
# electrodes = [
#     {'center': (-x_sep, -20), 'dims': (width, height), 'voltage': 1},   # rf-left
#     {'center': ( x_sep, -20), 'dims': (width, height), 'voltage': -1},     # ground-right
#     {'center': (-outer_x_sep, -20), 'dims': (2*width, height), 'voltage': -1},     # rf-right
#     {'center': ( outer_x_sep, -20), 'dims': (2*width, height), 'voltage': 1}    # ground-right
# ]

# Define hyperbolic electrode centers relative to center (0,0)
x_sep = 0
y_sep = 0
a = 280
b = 280
electrodes = [
    {'center': (-x_sep,  y_sep), 'dims': (a, b), 'theta': -45, 'branch': 'positive', 'voltage': 1},   # top-left
    {'center': ( x_sep,  y_sep), 'dims': (a, b), 'theta': 45, 'branch': 'positive', 'voltage': -1},     # top-right
    {'center': (-x_sep, -y_sep), 'dims': (a, b), 'theta': -45, 'branch': 'negative', 'voltage': -1},     # bottom-left
    {'center': ( x_sep, -y_sep), 'dims': (a, b), 'theta': 45, 'branch': 'negative', 'voltage': 1}    # bottom-right
]

# Initialize potential array and fixed mask
V = np.zeros((ny, nx))
fixed = np.zeros_like(V, dtype=bool)

# Create electrode masks and set fixed voltages
for electrode in electrodes:
    # Rod electrodes
    # cx, cy = electrode['center']
    # mask = create_circular_electrode_mask(cx, cy, radius, X, Y)
    
    # Rectangular electrodes
    # cx, cy = electrode['center']
    # width, height = electrode['dims']
    # mask = create_rectangular_electrode_mask(cx, cy, width, height, X, Y)
    
    # Hyperbolic electrodes
    cx, cy = electrode['center']
    a, b = electrode['dims']
    theta = electrode['theta']
    branch = electrode['branch']
    mask = create_rotated_hyperbolic_electrode_mask(cx, cy, a, b, theta, X, Y, branch=branch)
    
    # Apply mask and update electrode with mask values. Will be used to inforce boundary conditions
    electrode['mask'] = mask
    V[mask] = electrode['voltage']
    fixed[mask] = True

# Solve for potential
V = solve_potential(nx, ny, electrodes, fixed, V, max_iter=10000, tol=1e-5)

# Compute electric field
Ex, Ey = compute_electric_field(V, dx=dx, dy=dx)

# %% View just electrode geometry

fig, axes = plt.subplots(1, 1, figsize=(8, 8))

for e in electrodes:
    plot_mask_outline(axes, e['mask'], X, Y)
plt.tight_layout()
plt.show()

# %% Plot potential

def plot_mask_outline(ax, mask, X, Y, linewidth=1.5, label=None):
    """
    Plots the outline of a boolean mask on a matplotlib axis.

    Parameters:
    - ax: Matplotlib axis
    - mask: 2D boolean array (same shape as X and Y)
    - X, Y: meshgrid coordinate arrays
    - color: line color
    - linewidth: line thickness
    - label: optional text label at center of mask
    """
    cs = ax.contour(X, Y, mask.astype(float), levels=[0.5], colors='red', linewidths=linewidth)

    if label is not None:
        # Place label at mask centroid
        ys, xs = np.where(mask)
        if len(xs) > 0:
            cx = X[0, xs].mean()
            cy = Y[ys, 0].mean()
            ax.text(cx, cy, label, color='red', ha='center', va='center', fontsize=8, fontweight='bold')

def plot_potential_and_field_side_by_side(V, Ex, Ey, X, Y, electrodes, step=4):
    """
    Creates side-by-side plots:
    - Left: Electric potential with electrode outlines
    - Right: Electric field vectors with same background and outlines

    Parameters:
    - V: 2D array of electric potential
    - Ex, Ey: 2D arrays of electric field components (Ex = -dV/dx, etc.)
    - X, Y: meshgrid arrays in microns
    - electrodes: list of dicts, each with 'mask' and optionally 'voltage'
    - step: arrow spacing for quiver
    """
    fig, axes = plt.subplots(1, 2, figsize=(20, 6), sharex=True, sharey=True)

    # --- Plot 1: Potential
    ax1 = axes[0]
    im1 = ax1.imshow(V, extent=[X.min(), X.max(), Y.min(), Y.max()],
                     origin='lower', cmap='viridis')
    fig.colorbar(im1, ax=ax1, label='Potential (V)')
    ax1.set_title("Electric Potential")
    ax1.set_xlabel("x (Î¼m)")
    ax1.set_ylabel("y (Î¼m)")

    # --- Plot 2: Electric Field Vectors
    ax2 = axes[1]
    im2 = ax2.imshow(V, extent=[X.min(), X.max(), Y.min(), Y.max()],
                     origin='lower', cmap='viridis')
    fig.colorbar(im2, ax=ax2, label='Potential (V)')
    ax2.quiver(X[::step, ::step], Y[::step, ::step],
               Ex[::step, ::step], Ey[::step, ::step],
               color='white', scale=100)
    ax2.set_title("Electric Field Vectors")
    ax2.set_xlabel("x (Î¼m)")

    # Overlay mask outlines on both plots
    for ax in axes:
        for e in electrodes:
            label = f"{e['voltage']} V" if 'voltage' in e else None
            plot_mask_outline(ax, e['mask'], X, Y, label=label)

    for ax in axes:
        ax.set_aspect('equal')

    plt.tight_layout()
    plt.show()

plot_potential_and_field_side_by_side(V, Ex, Ey, X, Y, electrodes)

# %%

from scipy.optimize import curve_fit
from scipy.interpolate import RegularGridInterpolator

def quadratic_fit(s, a, c):
    return a * s**2 + c

# Coords are X and Y positions
# ion_elc_dis is the ion to electrode distance
# pn_rs are the multipole coefficients for the "real" part
# pn_is are the multipole coefficients for the "imaginary" part (rotated)
def multipole_func(coords, ion_elc_dis, pn_rs, pn_is):
    if len(pn_rs) != len(pn_is):
        return None
    X, Y = coords
    # Include n_order terms
    n_order = len(pn_rs)
    potential = 0
    for i in range(n_order):
        multipole_expansion = (X + 1j * Y)**i
        pn_real_func = np.real(multipole_expansion)
        pn_imag_func = np.imag(multipole_expansion)
        potential += 1/ion_elc_dis**i * (pn_rs[i] * pn_real_func + pn_is[i] * pn_imag_func)
    return potential


def get_cut_along_angle(V, X, Y, angle_deg, length):
    angle_rad = np.deg2rad(angle_deg)
    s = np.linspace(-length, length, 2 * length + 1)  # 1 Î¼m steps
    x_cut = s * np.cos(angle_rad)
    y_cut = s * np.sin(angle_rad)

    interp_func = RegularGridInterpolator((Y[:,0], X[0,:]), V)
    points = np.vstack((y_cut, x_cut)).T  # order (y, x)
    V_cut = interp_func(points)
    return s, V_cut, x_cut, y_cut

# Test Multipole expansion function is correct
# test_pot = multipole_func((X, Y), 30, [0,0,1], [0,0,1])
# fig, axes = plt.subplots(1, 1, figsize=(14, 6), sharex=True, sharey=True)
# # --- Plot 1: Potential
# ax1 = axes
# im1 = ax1.imshow(test_pot, extent=[X.min(), X.max(), Y.min(), Y.max()],
#                   origin='lower', cmap='viridis')
# fig.colorbar(im1, ax=ax1, label='Potential (V)')
# ax1.set_title("Electric Potential")
# ax1.set_xlabel("x (Î¼m)")
# ax1.set_ylabel("y (Î¼m)")

# %% Fit potential to multipole expansion coefficients

# Pick the number of poles
def multipole_fit(coords, p0_r, p1_r, p2_r, p3_r, p4_r, p5_r, p6_r, p7_r, p8_r, p0_i, p1_i, p2_i, p3_i, p4_i, p5_i, p6_i, p7_i, p8_i):
    ion_elc_dis = 200 * np.sqrt(2)
    pn_rs = [p0_r, p1_r, p2_r, p3_r, p4_r, p5_r, p5_r, p6_r, p7_r, p8_r]
    pn_is = [p0_i, p1_i, p2_i, p3_i, p4_i, p5_i, p6_i, p6_i, p7_i, p8_i]
    return multipole_func(coords, ion_elc_dis, pn_rs, pn_is).flatten()

# Crop coordinates to just be near ion
startx = nx//2 - 15
endx = nx//2 + 16
starty = ny//2 - 15
endy = ny//2 + 16
X_cropped, Y_cropped = X[starty:endy, startx:endx], Y[starty:endy, startx:endx]
V_cropped = V[starty:endy, startx:endx]
coords = (X_cropped, Y_cropped)
popt, pcov = curve_fit(multipole_fit, coords, V_cropped.flatten())
print(popt)
coef_mag = np.sqrt(popt[:9]**2 + popt[9:]**2)

np.set_printoptions(suppress=True)
print('Coefficients relative to quadrupole moment:\n', coef_mag[2:] / coef_mag[2])
np.set_printoptions(suppress=False)
fit = np.reshape(multipole_fit((X_cropped, Y_cropped), *popt), X_cropped.shape)
guess = np.reshape(multipole_fit((X_cropped, Y_cropped), *[0,0,0.0,0,0,0,0,0,0,0,0,-0.82,0,0,0,0,0,0]), X_cropped.shape)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
# --- Plot 1: Potential
ax1 = axes[0]
im1 = ax1.imshow(V_cropped, extent=[X_cropped.min(), X_cropped.max(), Y_cropped.min(), Y_cropped.max()],
                 origin='lower', cmap='viridis')
fig.colorbar(im1, ax=ax1, label='Potential (V)')
ax1.set_title("Electric Potential")
ax1.set_xlabel("x (Î¼m)")
ax1.set_ylabel("y (Î¼m)")
# --- Plot 2: Fit Potentials
ax2 = axes[1]
im2 = ax2.imshow(guess, extent=[X_cropped.min(), X_cropped.max(), Y_cropped.min(), Y_cropped.max()],
                 origin='lower', cmap='viridis')
fig.colorbar(im2, ax=ax2, label='Potential (V)')
ax2.set_title("Electric Potential Fit")
ax2.set_xlabel("x (Î¼m)")

print('LS', np.sqrt(np.sum((guess - V_cropped)**2)))
print('LS', np.sqrt(np.sum((fit - V_cropped)**2)))

# %%

length = 100  # microns half-length of cut

angles = [-45, 45]
plt.figure(figsize=(5, 10))

# Plot the 2D potential with diagonal cut lines
plt.subplot(3,1,1)
extent = [X.min(), X.max(), Y.min(), Y.max()]
im = plt.imshow(V, origin='lower', extent=extent, cmap='viridis')
plt.colorbar(im, label='Potential (V)')
plt.xlabel('x (Î¼m)')
plt.ylabel('y (Î¼m)')
plt.title('Potential with Diagonal Cuts')

# Plot cut lines on the potential plot
for angle in angles:
    angle_rad = np.deg2rad(angle)
    # line endpoints for full plot extent
    L = max(X.max() - X.min(), Y.max() - Y.min())
    s_line = np.linspace(-L/2, L/2, 500)
    x_line = s_line * np.cos(angle_rad)
    y_line = s_line * np.sin(angle_rad)
    plt.plot(x_line, y_line, 'w--', lw=2, label=f'{angle}Â° cut')
plt.legend()

# Plot cuts and fits
for i, angle in enumerate(angles):
    s, V_cut, x_cut, y_cut = get_cut_along_angle(V, X, Y, angle, length)
    
    # Fit quadratic
    guess = [1e-5]
    popt, pcov = curve_fit(quadratic_fit, s, V_cut)
    a_fit, c_fit = popt
    
    # Center to electrode distance for 4 rod trap
    # electro_dis = np.sqrt(2) * half_sep - radius
    # Center to electrode distance for rectangular trap
    # e_x, e_y = x_sep - width/2, y_sep - height/2
    # electro_dis = np.sqrt(e_x**2 + e_y**2)
    # electrode distance for hyperbolic trap
    electro_dis = 50
    print('Geometric factor', np.round(np.abs(a_fit * electro_dis**2), 4))
    
    # Plot cut data and fit
    plt.subplot(3,1,i+2)
    plt.plot(s, V_cut, 'b.', label='Data')
    plt.plot(s, quadratic_fit(s, *popt), 'r-', label=f'Fit: a={a_fit:.3e} V/Î¼mÂ²')
    plt.title(f'Cut at {angle}Â°')
    plt.xlabel('Distance s (Î¼m)')
    plt.ylabel('Potential V (V)')
    plt.grid(True)
    plt.legend()
    print(f"Angle {angle}Â° fit: a = {a_fit:.3e} V/Î¼mÂ², offset = {c_fit:.3e} V")

plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'matplotlib'